In [1]:
import h5py as h5
import numpy as np

# Solid Si: computing electron density with AFQMC

TODO: Incldue the PSP file, or point to where one can get the file.

## Introduction

TODO: cover the following points:

- AFQMC can compute general observables, $\hat{O}$ in addition to the energy
- for $\hat{O}$ which commute with $\hat{H}$, we can use mixed-estimators - for example, the Hamiltonian commutes with itself, so we can safely use a mixed-estimator.
- for observables which do not commute with the Hamiltonian, we must evaluate expectation values using a stochastic representaton of the wavefunction on "both sides"
- mixed estimators have the advantage of being computationally cheaper to evaluate without loosing accuracy
- a "pure" estimator is more expensive to evaluate because we must generate samples of the many-body wavefuncion on the left of the expation value *which are independent* of the samples on the right-hand side.
- in pracice, the back-propagation algorithm is employed in order to generate such samples efficiently.


In addition to computing the ground state energy and low-lying excited state energies, AFQMC is able to compute generic physical observables of the form:

$\hat{O}^{(1)} = \sum_{ij}\hat{c}^\dagger_i \hat{c}_j O^{(1)}_{ij} $

or

$\hat{O}^{(2)} = \sum_{ijkl}\hat{c}^\dagger_i \hat{c}^\dagger_j \hat{c}_k \hat{c}_l O^{(2)}_{ijkl} $.

For observables which commute with the Hamiltonian, a so-called mixed estimator can be used.
For example, the Hamiltonian trivially commutes with itself and the AFQMC energy is formally given by  

$ E =  \frac{\langle \Psi_T | \hat{H} e^{-\beta \hat{H}} | \Psi_I \rangle}{\langle \Psi_T | e^{-\beta \hat{H}} | \Psi_I \rangle}  $  

where $|\Psi_T\rangle$ is the trial wavefunction (used to control the fermionic sign/phase problem), and $\Psi_I$ is some initial 
wavefunction with $\langle\Psi_T|\Psi_I\rangle \neq 0$. This is referred to as a "mixed" estimator since the wavefunction on the 
left-hand side of the expression is not, in general, the same as the wavefunction on the right-hand side.
In the limit $\beta \rightarrow \infty$, $e^{-\beta \hat{H}} | \Psi_I \rangle \rightarrow | \Psi_n \rangle$ where $|\Psi_n \rangle$ is
an exact eigenstate of the Hamiltonian and $n$ is a generic quantum number index.
In that limit, $ E = E_n \frac{\langle \Psi_T | \Psi_n \rangle}{\langle \Psi_T | \Psi_n \rangle}$ 
and the mixed estimator produces the many-body energy $E_n$.

However, for observables that do not commute with the Hamiltonian, such as the density operator, a so-called "pure" estimator must be used.
(the points are: 
1. we need to back-propagate, which introduces additional parameters 
2. observables (specifically the electron density) are expressed in terms of the 1-rdm/Green's function
we need to equilibrate the 1-rdm in the B.P. run parameters, and analyzing the 1-rdm is a key step in computing general observables.
)

TODO: write down the pure estimator


TODO: We need to break down the combination of forward- and back-propagation

### References:

[1] 

## Tutorial




# Initial DFT using Quantum Espresso

We will use Quantum Espresso (QE) to perform initial DFT calculations.
The goal is to both generate a basis of orbitals, and to generate a trial wavefunction.
In addition to running the PWSCF program, we will use a few of QE's post-processing tools. 
Sample input files for each are inlcuded below.
The commands invoking the QE executables can either be run locally or in a runscript on a cluster.

## Initial Self-Consistent DFT:

```bash
mpirun pw.x -inp si_qe.in > scf.out
```

### for pw.x: si_qe.in

```
&control
   calculation='scf'
    restart_mode='from_scratch',
    pseudo_dir = './',
    outdir='./tmp/',
    prefix='Si-fcc',
    verbosity='high' # turn off nonsymmporphic
 /

 &system
    ibrav=  0,
    celldm(1)=10.263087,
    nat= 2,
    ntyp= 1,
    nbnd= 48,
    ecutwfc =25.0,
    input_dft='LDA',
    force_symmorphic=.true.,
    nr1=24,
    nr2=24,
    nr3=24 
 /

 &electrons
    diago_full_acc = .true.,
    conv_thr = 1.0e-10
 /

ATOMIC_SPECIES
 Si 28  14_Si_LDA_25Ry_SRL.UPF

K_POINTS automatic
1 1 1 0 0 0 


ATOMIC_POSITIONS crystal
 Si   -0.1250000000000000  -0.1250000000000000  -0.1250000000000000
 Si    0.1250000000000000   0.1250000000000000   0.1250000000000000

CELL_PARAMETERS
0.00  0.50  0.50
0.50  0.00  0.50
0.50  0.50  0.00
```

For AFQMC, we will typically want a relatively number of bands compared to a typical DFT calculation.
If desired, we can run an "nscf" calculation with more bands.
If we increase the number of basis set orbitals (i.e. bands), the AFQMC energy will evenually converge.
Below is a Figure that demonstrates this.

(Add an AFQMC E vs Nbasis convergence plot here)





## Run QE post-processing steps

```bash
mpirun pp.x < pp_vsc.inp > pp_vsc.out
mpirun pp.x < pp_vltot.inp > pp_vltot.out
mpirun pw2bgw.x < pw2bgw.inp > pw2bgw.out
```

### for pp_vsc.x : pp_vsc.inp

```
&inputpp

outdir = "./tmp",
prefix = "Si-fcc",
filplot = "VSC",
plot_num = 1

/
```

### for pp_vltot.x : pp_vltot.inp

```
&inputpp

outdir = "./tmp",
prefix = "Si-fcc",
filplot = "VLTOT",
plot_num = 2

/
```

### for pw2bgw.x : pw2bgw.inp

```
&input_pw2bgw

prefix = 'Si-fcc'
outdir = './tmp'
real_or_complex = 2
wfng_flag = .false.

rhog_flag = .false.

vxcg_flag = .false.
vxcg_file = 'VXC'

vkbg_flag = .true.
vkbg_file = 'VKB'

/
```

# Generate a 2nd-quantized Hamiltonian and Trial Wavefunction for AFQMC

CCQ's GW code, AIMB, is also able to generate a Hamiltonian in a Kohn-Sham orbital basis that can be directly used in QMC@FI/AFQMC code.
A complete descrption of AIMB and all of its capabilities is beyond the scope of this tutorial.
Here we focus only on the features necessary to generate input for AFQMC.

Below is a sample input file for AIMB. 
We will break each section down below.
Note that each "section" of input includes a "name" field which can be used to 
reference that specific section later in the input file.
This allows multiple sections of the same type to be included in the input if desired.
The presence of some input file sections tells AIMB to carry out specific calculations.
Here, the "integrals" and the "hamiltonian" sections tell AIMB to calculte integrals for later use
and the generate a Hamiltonian, respectively.

```json
{
  "mean_field":{
    "name": "mf",
    "type": "qe",
    "prefix": "Si-fcc",
    "outdir": "./tmp",
    "vkb": "./tmp/VKB",
    "vltot": "./VLTOT",
    "vsc": "./VSC" 
  },
  "integrals" : {
    "name":"hamilt",
    "mean_field":"mf",
    "type": "cholesky",
    "output" : "hamil.bdft.h5",
    "write_type" : "single",
    "thresh": 1e-5
  },
  "hamiltonian" : {
    "mean_field":"mf",
    "add_wavefunction" : "default",
    "output": "hamil.bdft.h5"
  }
}
```




### the "mean_field" section

The `mean_field` section of the input file is used to specify to AIMB which mean-field result to use.
We've named the section "mf".

```json
"mean_field":{
    "name": "mf",
    "type": "qe",
    "prefix": "Si-fcc",
    "outdir": "./tmp",
    "vkb": "./tmp/VKB",
    "vltot": "./VLTOT",
    "vsc": "./VSC" 
  },
```

The `"type": "qe"`` field indicates that we are using a Quantum Espresso result as input.
The `"prefix"` and `"outdir"` fields must be set to the same values as in the `&control` card of the Quantum Espresso input file.
Finally, "vltot"`, `"vsc"`, and "vkb"` must be set to the path (including filename) where  "VLTOT", "VSC", and "VKB" were generated
in the precedding steps.


### the "integrals" section

The integrals section specifies how AIMB will handle integrals internally.
We've named it "hamilt".

```
"integrals" : {
    "name":"hamilt",
    "mean_field":"mf",
    "type": "cholesky",
    "output" : "hamil.bdft.h5",
    "write_type" : "single",
    "thresh": 1e-5
  },
```

The "mean_field" field should be set to the *name* a mean_field block that will provide an orbital basis. 
In the case, we're using "mf".
The "type" is used to specify what type of electron-electron interaction intergrals to use.
One of AIMB's key features is its tensor hyper-contraction (THC) implementation that allows it to handle very
large system sizes.
Here, we are using "cholesky" integrals instead since we are using a small system.
"output" is used to specify the name of the output file for AIMB.
"thresh" is a numerical threshold used for the integrals.
For Cholesky, the error in the two-body *integrals* (not the two-body energy) is bounded by the threshold.

### the "hamiltonian" section

The Hamiltonian section indicates to AIMB that it should generate a Hamiltonian.
Again, we must provide a "mean_field" section which will be used as a basis.

```json
"hamiltonian" : {
    "mean_field":"mf",
    "add_wavefunction" : "default",
    "output": "hamil.bdft.h5"
  }
```

Questions:
1. what options exist for add_wavefunction? I assume that the default is to use the DFT Slater determinant.
2. why don't we need to give "hamiltonian" an "integrals" argument?

# AFQMC ground state energy and electron density

## Running AFQMC

### Generate an Input file

After running the steps above, there should be an HDF5 file called `afqmc.h5` in the same directory as this notebook.
To run easyAF, we will also need a json input file.
We have provided a command line tool that can generate a json input file based on the contents of `afqmc.h5`.
This can be invoked as follows (after installing the Python tools).

```bash
$ write_afqmc_json -i afqmc.h5 -b 400
```

which will generate a file called `afqmc.json` with the following contents:

TODO: update for si, add in Back-Propagation

Alos, explain how "estimators" work in the code. i.e. multiple estimators can be included in the input file, etc.

```json

```


In [10]:
! write_afqmc_json -i afqmc.h5 -b 400


### Invoke the AFQMC executable

we are now ready to invoke the AFQMC executalbe. 
In general, AFQMC will be run on a comuting cluster across several nodes / GPUs.
A general guide to running AFQMC on arbitrary clusters is beyond the scope of this tutorial.
However, we provide a basic example of a Slurm script below.
We note that this is small system for AFQMC, so we are running only on CPUs with few nodes.

```bash
#!/bin/bash -l
#SBATCH -J AFQMC_si

#! Number of MPI ranks (= tasks for Slurm)
#SBATCH --ntasks=160
#SBATCH --time=1:00:00

# 1. perform environment setup
export AFQMC_PATH=/path/to/afqmc/exec

# 2. Launch MPI code...
srun --cpu-bind=cores $AFQMC_PATH/bin/qmcapp --filenames afqmc.json &> afqmc.out
```

In [11]:
# For a Binder-hosted notebook, we could do this!
! sbatch runscript.sh

Submitted batch job 3021607


## Analysis

Finally, we must perform a statistical analysis of the AFQMC output.
We describe how to do this in a separate tutorial which can be found [here](). 
 
TODO: link the analysis tutorial.

In [ ]:
! scalar_stats qmc.s000.scalar.dat -x time -e 5.0

EnergyEstim__nume_real -109.089320 +/-   0.000615 4.56  5.0/40.0


Now, we want to analyze the output from the back-propagation estimator. We describe how to do this in a separate tutorial which can be found [here](). 
 
TODO: link the analysis tutorial.

In [ ]:
# TODO: move to an analysis-specific example once finished

# 1. raw AFQMC output to a useable format

## Visualize the results

TODO: learn Paul's tools and incorporate - then explain how to use them here.

In [ ]:
from pathlib import Path

from afqmctools.utils.qe_utils import read_qe_orbitals
from stats import stat_h5


def get_metadata(fstat, path='Metadata'):
    # copied from qharv
    meta = dict()
    with h5.File(fstat, 'r') as fp:
        for k, v in fp[path].items():
            meta[k] = v[()]
    return meta

def get_bp_taus(fstat):
    # copied from qharv
    sym_md = get_metadata(fstat)
    bp_md = get_metadata(fstat, 'Observables/BackPropagated/Metadata')
    dt = sym_md['Timestep']
    nsteps = bp_md['BackPropSteps']
    taus = nsteps*dt
    return taus

def get_afqmc_rdm_samples(rdm_fname,nequil=0):
    """  
    Read 1-rdm from AFQMC format
    """
    rdm_fname = Path(rdm_fname)
    taus = get_bp_taus(rdm_fname)

    # 1. Read from hdf5
    with h5.File(rdm_fname,'r') as f:
        dm_map = {'taus': taus}
        for iav, tau in enumerate(taus):
            name = 'a%d' % iav
            dm, de = stat_h5.afobs(
                f,
                'FullOneRDM',
                nequil,
                numer='one_rdm',
                iav=iav
            )
            dm_map[name] = {'dm_mean': dm, 'dm_error': de}
    return dm_map

def _line_search_bool(xvals,yvals,zvals,rho,drho):
    
    match_xy = np.isclose(xvals,yvals)
    match_yz = np.isclose(yvals,zvals)
    
    line_inds = np.logical_and(match_xy,match_yz)

    r = xvals[line_inds]
    rho_line = rho[line_inds]
    drho_line = drho[line_inds]
    
    return r,rho_line,drho_line


def get_line_cut(rho,grid,drho=None,coords_first=True):
    """
    for now, search along x=y=z
    """

    if drho is None:
        drho = np.zeros_like(rho)

    if coords_first:
        xvals = grid[0,:]
        yvals = grid[1,:]
        zvals = grid[2,:]
    else:
        xvals = grid[:,0]
        yvals = grid[:,1]
        zvals = grid[:,2]

    r,line_rho,line_drho = _line_search_bool(
        xvals=xvals,
        yvals=yvals,
        zvals=zvals,
        rho=rho,
        drho=drho
        )
    return r,line_rho,line_drho

def compute_rho_v2(Gab,phi_kar):
    """
    Compute the k-point averaged rho(r) given the 1-body reduced density matrix (Gab)
       and the basis orbitals in real-space representation. The density
       is represented on the same spatial grid as the basis orbitals.

    Inputs:
    - Gab:np.ndarray with shape n_basis x n_basis - 1-body reduced density matrix (a.k.a 
                                                    the 1-body Green's function)
    - phi_kar:np.ndarray with shape n_kpoint x n_basis x n_grid_points
    """
    nkpts = phi_kar.shape[0]
    for k in range(nkpts):
        phi_ar = phi_kar[k]
        Gbar = np.matmul(Gab,phi_ar)
        if 'rho' in locals():
            rho += np.einsum('ar,ar->r',phi_ar.conj(),Gbar)
        else:
            rho = np.einsum('ar,ar->r',phi_ar.conj(),Gbar)
    return rho

def analyze_afqmc_density(qe_prefix,qe_path,rdm_filename,rdm_path,outname=None,Neq=0,iav=0):
    print("Analyzing CCQ AFQMC (AIMB converter) density")

    if outname is None:
        outname = "ccq_afqmc_line_cut.dat"

    print("Reading QE orbitals")
    r,orbitals_r,M = read_qe_orbitals(
        prefix=qe_prefix,
        path=qe_path
    )

    dm_map = get_afqmc_rdm_samples(rdm_path/rdm_filename)

    #avg_rdm1,delta_rdm1 = average_afqmc_rdm(dm_map, Neq=Neq) # TODO: Remove all refs to this, it is not
    avg_rdm1 = dm_map[f"a{iav}"]["dm_mean"][0]
    delta_rdm1 = dm_map[f"a{iav}"]["dm_error"][0] 

    rho_afqmc = 2*compute_rho_v2(avg_rdm1,orbitals_r).real
    # HACK: the following is (technically) wrong I suspect - should probably compute rho for many samples of the rdm1,
    #             and get the stochastic uncertainty matrix from an alaysis of those samples in real space.
    delta_rho_afqmc = 2*compute_rho_v2(delta_rdm1,orbitals_r).real

    

    print("Normalizing...")
    norm_fac_rho_afqmc = np.sum(rho_afqmc)
    print(f"normalization factor = {norm_fac_rho_afqmc}",level=1)
    rho_afqmc = rho_afqmc / norm_fac_rho_afqmc
    delta_rho_afqmc = delta_rho_afqmc / norm_fac_rho_afqmc

    r_afqmc,rho_afqmc_line,drho_afqmc_line = get_line_cut(
            grid=r,
            rho=rho_afqmc.real,
            drho=delta_rho_afqmc.real
        )

    np.savetxt(outname,np.array((r_afqmc,rho_afqmc_line,drho_afqmc_line)).T)


analyze_afqmc_density(
    
)

In [1]:
import pyvista as pv
pv.set_jupyter_backend('trame')


def isosurface(
        grid_in,
        rho,
        n=24
    ):

    x_min = np.min(grid_in[0,:])
    y_min = np.min(grid_in[1,:]) 
    z_min = np.min(grid_in[2,:]) 

    x_max = np.max(grid_in[0,:])
    y_max = np.max(grid_in[1,:]) 
    z_max = np.max(grid_in[2,:]) 

    grid = pv.ImageData(
        dimensions=(n, n, n),
        spacing=(x_max/n, y_max/n, z_max/n),
        origin=(x_min, y_min, z_min),
    )
    
    values = rho
    mesh = grid.contour(
        15,
        values,
        method='marching_cubes'
        )
    mesh.plot(
        scalars=None,
        smooth_shading=True,
        specular=0.5,
        cmap="twilight",
        show_scalar_bar=False
    )

def _format_block(block):
    x,y,z,rho,drho = np.loadtxt(block,unpack=True)
    return np.array([x,y,z]),rho,drho


def read_pwafqmc_density(fname="density_grid_qmc.dat"):

    with open(fname) as f:
        lines = f.readlines()
        
        # initialize
        _meshes = []
        _rhos = []
        _drhos = []

        idx = 0
        buff = lines[idx:]
        
        while len(buff) > 0:
            idx = buff.index('\n')
            if idx > 0:
                _mesh,_rho,_drho = _format_block(buff[:idx])
                if _mesh.size != 0:
                    _meshes.append(_mesh)
                if _rho.size != 0:
                    _rhos.append(_rho)
                if _drho.size != 0:
                    _drhos.append(_drho)

            buff = buff[idx+1:]


    mesh = np.concatenate(_meshes,axis=1)
    rho = np.concatenate(_rhos)
    drho = np.concatenate(_drhos)

    return mesh,rho,drho

class ElectronDensity:

    def __init__(self,fname=None,type='ccq') -> None:

        if type.lower() == "ccq":
            self.type = "ccq"
            read_density = read_ccq_density
        elif type.lower() == "pwafqmc":
            self.type = "pwafqmc"
            read_density = read_pwafqmc_density
        else:
            raise ValueError("Invalid electron density type")

        self.grid,self.data,self.data_error = read_density(fname)


    def text_dump(self,outname='rho.txt'):
        np.savetxt(
            outname, 
            self.grid,
            fmt='%14.13f')


rho_pwafqmc = ElectronDensity(
    fname="density_grid_qmc.dat",
    type="pwafqmc"
    )
    
isosurface(grid_in=rho_pwafqmc.grid,rho=rho_pwafqmc.data,isoval=0.04)


NameError: name 'np' is not defined